# RADAR (biến thể XLM-R-Large) — game paraphraser ⇄ detector qua PPO

Hu et al., NeurIPS 2023, [arXiv:2307.03838](https://arxiv.org/abs/2307.03838).

| | |
|---|---|
| **Reward** (Eq. 1) | `R(x_p, φ) = D_φ(x_p)` = xác suất **human**. Không có gì thêm |
| **Paraphraser** (Eq. 2) | cppo-ep = clipped PPO + **entropy penalty γ=0.01**; advantage = chuẩn hoá R trên toàn buffer |
| **Detector** (Eq. 3) | `L_D = −E[log D(x_h)] + λE[−log(1−D(x_m))] + λE[−log(1−D(G(x_m)))]`, **λ=0.5** |
| **Chọn checkpoint** | `φ_best` theo AUROC(val), `σ_best` là paraphraser **ghép cặp** với nó (Algorithm 1 dòng 27-28) |

## Ba điều phải hiểu trước khi chạy

**1. `Dφ` khởi tạo từ `roberta-large` PRETRAINED, KHÔNG load checkpoint supervised.**
Đúng Algorithm 1 dòng 7, và giữ hai nhánh độc lập — không model nào trong bảng 6 đứng trên vai model khác.

**2. Cold start là hành vi ĐÚNG, không phải bug.** Vài chục outer step đầu `Dφ` còn ngẫu nhiên → reward
gần như hằng số → `Gσ` đi lang thang. Nhưng **Eq. 3 không phụ thuộc reward chút nào** — `Dφ` vẫn học đều
từ `(x_h, x_m)` ngay từ step 1. Khi `Dφ` đủ phân biệt thì reward mới có phương sai.
→ Guard `if R.std() < EPS: skip PPO` là **bắt buộc**: `A = (R−mean)/std` không xác định khi `std≈0`,
và nó sẽ nổ thành nhiễu khuếch đại đủ để phá `Gσ`.

**3. Đơn vị sinh = CÂU, reward chấm trên DOCUMENT.**

```
x_m (~275 từ ≈ 12 câu)
  → paraphrase TỪNG câu (humarin giới hạn 128 token in/out)
  → ghép lại theo đúng thứ tự  → x_p
  → reward = 1 − P_Dφ(AI | x_p)   ← chấm trên DOCUMENT ĐẦY ĐỦ, 512 token
  → gán reward đó cho MỌI câu của document (reward cấp trajectory, chuẩn RLHF)
```

Giữ số câu ⇒ giữ `word_count` ⇒ **không tạo shortcut độ dài** cho detector. Nếu paraphrase cả doc với
`max_new_tokens` nhỏ thì `x_p` ngắn hệ thống so với `x_h`/`x_m` → detector học độ dài, AUROC đẹp giả.


> **Biến thể notebook này**: đổi `Dφ` sang **XLM-RoBERTa-large** thay vì `roberta-large`, để so sánh với nhánh supervised XLM-R-large (`train_04_xlm_roberta.ipynb`) và `train_adv_01_greater_xlm.ipynb` — cả 3 cùng backbone, khác quy trình train. `PARAPHRASER` giữ nguyên (không phụ thuộc detector). Vẫn giữ nguyên toàn bộ Eq. 1-3 và Algorithm 1 của RADAR — chỉ đổi kiến trúc `Dφ`, không đổi phương pháp.

## Khác biệt so với 2 notebook AL kia

**1 GPU** (không DDP) · **1 seed** · `Gσ` **fp32** (T5 tràn số ở fp16, T4 không có bf16) ·
`transformers` **ghim 4.45.2** (khác 4.57.6 của supervised — trl 0.11 chưa hỗ trợ 4.57) ·
**không fix cứng số epoch**, để early-stop theo AUROC(val) quyết · **có nhánh resume** qua session.

> Chạy `preflight_paraphraser.ipynb` **trước** để chốt `PARAPHRASER` + `PARA_UNIT`.

## 1. Cài đặt — ghim CHÍNH XÁC, không dùng `*`

`trl ≥ 0.12` là PPOv2 với API khác hẳn → **không dùng**. Bộ combo dưới đây là cho API cổ điển
`PPOTrainer.step(queries, responses, rewards)`.

> Sau khi cài **phải Restart kernel** rồi chạy cell verify.

In [ ]:
!pip install -q "transformers==4.45.2" "trl==0.11.4" "accelerate==0.34.2" "tokenizers<0.21" sentencepiece sentence-transformers bitsandbytes nltk
import nltk
for p in ["punkt", "punkt_tab"]:
    nltk.download(p, quiet=True)
print("\n>>> RESTART KERNEL roi chay cell verify ben duoi <<<")

In [ ]:
# ============ VERIFY COMBO -- chay TRUOC khi ton compute ============
import transformers, trl, accelerate, torch, inspect
print("transformers", transformers.__version__, "| trl", trl.__version__,
      "| accelerate", accelerate.__version__, "| torch", torch.__version__)
assert transformers.__version__.startswith("4.45"), "phai la transformers 4.45.x"
assert trl.__version__.startswith("0.11"),          "phai la trl 0.11.x (>=0.12 la PPOv2, API khac)"

from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead, create_reference_model
sig = inspect.signature(PPOTrainer.step)
print("PPOTrainer.step", sig)
assert "queries" in sig.parameters and "responses" in sig.parameters and "scores" in sig.parameters, \
    "API step() khong khop -> sai version trl"

from trl.core import masked_mean, entropy_from_logits          # can cho entropy penalty gamma
print("trl.core.masked_mean / entropy_from_logits: OK")
print("GPU:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("\nCOMBO OK -- co the chay train_radar.py")

In [ ]:
%%writefile train_radar.py
# ============================================================================
#  RADAR bien the XLM-R-Large -- game paraphraser <-> detector qua PPO.  1 GPU, 1 seed.
#  Dphi = xlm-roberta-large (khong phai roberta-large) -- xem markdown cell dau ve ly do.
#  Chay : python train_radar.py
#  Smoke: SMOKE=1 python train_radar.py
#  Resume tu dong: co ckpt_resume/ thi chay tiep, khong thi bat dau tu 0.
# ============================================================================
import os, json, math, random, time, warnings
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import nltk
from sklearn.metrics import roc_auc_score, average_precision_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup)
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead, create_reference_model
from trl.core import masked_mean, entropy_from_logits

warnings.filterwarnings("ignore")

# ---------------------------- CONFIG ----------------------------------------
RUN_NAME    = "radar_xlm"
DETECTOR    = "xlm-roberta-large"                          # Algorithm 1 dong 7: PRETRAINED (bien the XLM)
PARAPHRASER = "humarin/chatgpt_paraphraser_on_T5_base"     # deviation, xem plan §10.1
PARA_UNIT   = "cau"                                        # "cau" | "cua_so"  <- chot boi preflight
PREFIX      = "paraphrase: "

SEED        = int(os.environ.get("RUN_SEED", "42"))
SMOKE       = os.environ.get("SMOKE", "0") == "1"

# --- so cua PAPER (KHONG doi) ---
PPO_LR          = 1e-5     # AdamW linear decay
DET_LR          = 1e-5     # trung luon voi supervised
RADAR_LAMBDA    = 0.5      # Eq. 3 (ablation: 0.6 -- paper §4.6 cho thay tot ca 2 phia)
GAMMA_ENT       = 0.01     # entropy penalty cua cppo-ep (Eq. 2)
TOP_K, TOP_P    = 50, 0.95
INIT_KL_COEF    = 0.2
KL_TARGET       = 6.0

# --- so cua HA TANG ---
PPO_BATCH       = 16 if SMOKE else 256    # so CAU moi lan ppo.step()
PPO_MINI_BATCH  = 4  if SMOKE else 16
PPO_EPOCHS      = 4
DOCS_PER_STEP   = 4  if SMOKE else 24     # document moi outer step
DET_TRIPLETS    = 10                      # 10 bo ba -> batch 30 text, ty le 1:1:1 CHINH XAC
MAX_OUTER_STEPS = 3  if SMOKE else 3000   # de rong; EARLY-STOP moi la thu quyet dinh
MAX_LEN         = 512                     # detector, dong bo supervised
MAX_INPUT_TOK   = 128                     # gioi han cua humarin
MAX_NEW_TOKENS  = 96
MIN_SENT_WORDS  = 5
WINDOW_WORDS    = 90
GEN_BATCH       = 32
EVAL_EVERY      = 2  if SMOKE else 20     # outer step -- ha tu 40 de early-stop phan ung nhanh hon (~3h thay vi ~6h)
SAVE_EVERY      = 2  if SMOKE else 20
EARLY_STOP_EVALS = 6                      # so lan eval khong cai thien thi dung
REWARD_STD_EPS  = 1e-3
TARGET_FPR      = 0.01
# ── Bản dataset dùng để train/eval ────────────────────────────────────
#   "deconf2"    = đã khử confound bề mặt v2 (02c_deconfound_v2.ipynb)  <- ĐANG DÙNG
#                  canonicalize + truncate_pair_sentences + lọc notation_gap 85%.
#                  Đo bằng shortcut_audit.py (22 đặc trưng, 4 tầng):
#                    tầng process (dấu vết quy trình) 0.5055  <- cổng cứng, PASS
#                    tổng 22 đặc trưng               0.8909
#                  7.025 cặp train (85% của 8.265) — lọc bỏ những cặp mà bản AI lược
#                  ký hiệu/số/tên riêng của bài gốc nhiều nhất.
#   "deconf"     = bản 02b_deconfound.ipynb. ĐỪNG dùng cho bản chính thức:
#                  02b tự báo "surface-GBM 0.6867, G6 PASS" nhưng con số đó chỉ đo 6
#                  đặc trưng, không có avg_word_len (0.7795) lẫn type_token_ratio
#                  (0.7727). Đo đủ 22: 0.9114. Và truncate_pair cắt thô theo từ làm
#                  ~nửa văn bản kết thúc giữa câu -> word_count 0.5000 + ends_terminal
#                  0.5236 khi đứng riêng nhưng GHÉP LẠI = 0.7264.
#   "normalized" = bản GỐC, CÒN shortcut: chỉ ĐẾM SỐ TỪ đã đoán đúng human/AI
#                  ở AUROC 0.7193 (surface-GBM 0.8080). G6 FAIL.
#
# PHẢI đặt TRONG train_ddp.py, KHÔNG phải trong notebook: file này chạy qua
# torchrun ở tiến trình RIÊNG nên biến của notebook không tới được. Đặt ngoài thì
# script chết NameError — và trước đó cell còn không chạy nổi, vì dòng
# `DATASET_TAG = ...` ở trên đẩy %%writefile khỏi dòng đầu, làm Jupyter hiểu nó
# thành line magic: "Line magic function `%%writefile` not found".
#
# LƯU Ý: Kaggle dataset phải chứa dataset_{train,val,test_ood_model,test_fp}_deconf2.parquet
#        VÀ matched_eval_{val,test_ood_model}_deconf2.parquet (đánh giá đối-shortcut).
#        Danh sách file cần upload: KAGGLE_UPLOAD.md ở gốc repo.
DATASET_TAG = "deconf2"

KAGGLE_INPUT = "/kaggle/input/datasets/doquangluccqh/halfdatadetector"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# 2 GPU: tach vai tro (khong phai DDP). Khong tensor nao co grad di qua ranh gioi GPU.
PARA_DEV = "cuda:0" if torch.cuda.is_available() else "cpu"
DET_DEV  = "cuda:1" if torch.cuda.device_count() > 1 else PARA_DEV
print("paraphraser -> %s | detector -> %s" % (PARA_DEV, DET_DEV), flush=True)

def find_data_dir():
    t = f"dataset_train_{DATASET_TAG}.parquet"
    for b in [Path(KAGGLE_INPUT), Path("output"), Path("."), Path("/kaggle/input")]:
        if b.exists():
            if (b / t).exists():
                return b
            h = list(b.rglob(t))
            if h:
                return h[0].parent
    raise FileNotFoundError(t)

DATA_DIR = find_data_dir()
OUT_DIR = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")) / \
          ("artifacts_%s_seed%d" % (RUN_NAME, SEED))
CKPT_DIR = OUT_DIR / "ckpt_resume"
OUT_DIR.mkdir(parents=True, exist_ok=True); CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR, flush=True)

# ---------------------------- DATA ------------------------------------------
# BO Step 1 cua Algorithm 1: khong can target LM de dung corpus M -- ta da co ai_text,
# lai con da-generator (5 family) nen manh hon paper o dung muc tieu cross-model.
def melt_pairs(df):
    h = pd.DataFrame({"text": df["human_text"], "label": 0, "paper_id": df["paper_id"],
                      "ai_degree": 0, "generator_family": "human"})
    a = pd.DataFrame({"text": df["ai_text"], "label": 1, "paper_id": df["paper_id"],
                      "ai_degree": df.get("ai_degree", 3),
                      "generator_family": df.get("generator_family", "unknown")})
    o = pd.concat([h, a], ignore_index=True)
    return o[o["text"].notna() & (o["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def melt_fp(df):
    has_ai = df["ai_text"].notna() if "ai_text" in df.columns else pd.Series(False, index=df.index)
    text = df["ai_text"].where(has_ai, df["human_text"]) if "ai_text" in df.columns else df["human_text"]
    o = pd.DataFrame({"text": text, "label": 0, "paper_id": df["paper_id"],
                      "ai_degree": df["ai_degree"].fillna(0).astype(int) if "ai_degree" in df.columns else 0,
                      "generator_family": "human"})
    return o[o["text"].notna() & (o["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def load_raw(n):
    p = DATA_DIR / ("dataset_%s_%s.parquet" % (n, DATASET_TAG))
    return pd.read_parquet(p) if p.exists() else None

train_raw = load_raw("train")                     # cap (x_h, x_m) -- H va M
pairs = train_raw[["human_text", "ai_text"]].dropna().reset_index(drop=True)
val_df = melt_pairs(load_raw("val"))
ood_raw = load_raw("test_ood_model"); ood_df = melt_pairs(ood_raw) if ood_raw is not None else None
fp_raw  = load_raw("test_fp");        fp_df  = melt_fp(fp_raw)   if fp_raw  is not None else None
if SMOKE:
    pairs = pairs.sample(n=min(64, len(pairs)), random_state=SEED).reset_index(drop=True)
    val_df = val_df.groupby("label", group_keys=False).sample(n=64, random_state=SEED)
    ood_df = None; fp_df = None
print("pairs(H,M): %s | val: %s" % (format(len(pairs), ","), format(len(val_df), ",")), flush=True)

# ---------------------------- MODELS ----------------------------------------
det_tok = AutoTokenizer.from_pretrained(DETECTOR)
det = AutoModelForSequenceClassification.from_pretrained(DETECTOR, num_labels=2).to(DET_DEV)
det.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
det.config.use_cache = False

para_tok = AutoTokenizer.from_pretrained(PARAPHRASER)
# fp32: ho T5 train bang bf16, chay fp16 tran so o feed-forward -> NaN. T4 khong co bf16.
policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(PARAPHRASER).to(PARA_DEV)
ref_policy = create_reference_model(policy)       # reference DONG BANG cho KL

try:
    import bitsandbytes as bnb
    det_opt = bnb.optim.AdamW8bit(det.parameters(), lr=DET_LR, eps=1e-8, weight_decay=0.01)
    ppo_opt = bnb.optim.AdamW8bit(filter(lambda p: p.requires_grad, policy.parameters()), lr=PPO_LR)
    print("optimizer: AdamW8bit (bitsandbytes)", flush=True)
except Exception as e:
    det_opt = torch.optim.AdamW(det.parameters(), lr=DET_LR, eps=1e-8, weight_decay=0.01)
    ppo_opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, policy.parameters()), lr=PPO_LR)
    print("optimizer: AdamW fp32 (bitsandbytes khong dung duoc: %s)" % e, flush=True)

det_steps_total = MAX_OUTER_STEPS * max(1, math.ceil(DOCS_PER_STEP / DET_TRIPLETS))
det_sched = get_linear_schedule_with_warmup(det_opt, 0, det_steps_total)   # linear decay, dung paper
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

class CppoEpTrainer(PPOTrainer):
    """Clipped PPO + Entropy Penalty (Eq. 2). trl 0.11 khong co he so entropy ->
    them -gamma*S(sigma) vao pg_loss."""
    def loss(self, old_logprobs, values, logits, vpreds, logprobs, mask, advantages, returns):
        pg_loss, vf_loss, stats = super().loss(old_logprobs, values, logits, vpreds,
                                               logprobs, mask, advantages, returns)
        if GAMMA_ENT > 0:
            ent = masked_mean(entropy_from_logits(logits), mask)
            pg_loss = pg_loss - GAMMA_ENT * ent
        return pg_loss, vf_loss, stats

ppo_cfg = PPOConfig(model_name=PARAPHRASER, learning_rate=PPO_LR,
                    batch_size=PPO_BATCH, mini_batch_size=PPO_MINI_BATCH,
                    ppo_epochs=PPO_EPOCHS, gradient_accumulation_steps=1,
                    init_kl_coef=INIT_KL_COEF, adap_kl_ctrl=True, target=KL_TARGET,
                    seed=SEED, remove_unused_columns=False, log_with=None)
try:
    ppo = CppoEpTrainer(config=ppo_cfg, model=policy, ref_model=ref_policy,
                        tokenizer=para_tok, optimizer=ppo_opt)
    print("cppo-ep: entropy penalty gamma=%.3f DA BAT" % GAMMA_ENT, flush=True)
except Exception as e:
    ppo = PPOTrainer(config=ppo_cfg, model=policy, ref_model=ref_policy,
                     tokenizer=para_tok, optimizer=ppo_opt)
    print("CANH BAO: khong override duoc loss() -> chay PPO chuan, KHONG co entropy penalty "
          "(ghi vao bang Deviation). Ly do: %s" % e, flush=True)

GEN_KW = dict(do_sample=True, top_k=TOP_K, top_p=TOP_P, temperature=1.0,
              pad_token_id=para_tok.pad_token_id, eos_token_id=para_tok.eos_token_id,
              max_new_tokens=MAX_NEW_TOKENS)

# ---------------------------- HELPERS ---------------------------------------
def split_sentences(t):
    return [s for s in nltk.sent_tokenize(t) if s.strip()]

def split_windows(t, n=WINDOW_WORDS):
    out, cur, c = [], [], 0
    for s in split_sentences(t):
        w = len(s.split())
        if cur and c + w > n:
            out.append(" ".join(cur)); cur, c = [], 0
        cur.append(s); c += w
    if cur:
        out.append(" ".join(cur))
    return out

SPLIT = split_sentences if PARA_UNIT == "cau" else split_windows

@torch.no_grad()
def detector_scores(texts, batch=16):
    """P(AI). Quy uoc index 1 = AI (giong supervised) -> reward = 1 - score."""
    det.eval(); out = []
    for i in range(0, len(texts), batch):
        enc = det_tok(texts[i:i + batch], truncation=True, max_length=MAX_LEN,
                      padding=True, pad_to_multiple_of=8, return_tensors="pt").to(DET_DEV)
        with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            lg = det(**enc).logits
        out.extend(torch.softmax(lg.float(), -1)[:, 1].cpu().tolist())
    return np.array(out)

def paraphrase_batch(docs):
    """Tra ve (x_p list, queries list[Tensor], responses list[Tensor], doc_index list[int]).
    Moi CAU la 1 mau PPO; reward cua ca document se duoc broadcast xuong tung cau."""
    all_q_txt, owner, part_pos, parts_of = [], [], [], []
    for di, d in enumerate(docs):
        parts = SPLIT(d)
        parts_of.append(parts)
        for pi, p in enumerate(parts):
            if len(p.split()) >= MIN_SENT_WORDS:
                all_q_txt.append(PREFIX + p); owner.append(di); part_pos.append(pi)

    q_tensors, r_tensors = [], []
    for i in range(0, len(all_q_txt), GEN_BATCH):
        chunk = all_q_txt[i:i + GEN_BATCH]
        enc = para_tok(chunk, truncation=True, max_length=MAX_INPUT_TOK)["input_ids"]
        qs = [torch.tensor(e, device=PARA_DEV) for e in enc]
        rs = ppo.generate(qs, batch_size=len(qs), return_prompt=False, **GEN_KW)
        q_tensors += qs; r_tensors += list(rs)

    for k, (di, pi) in enumerate(zip(owner, part_pos)):
        txt = para_tok.decode(r_tensors[k], skip_special_tokens=True).strip()
        if txt:
            parts_of[di][pi] = txt         # giu nguyen thu tu -> giu do dai document
    return [" ".join(p) for p in parts_of], q_tensors, r_tensors, owner

def detector_update(x_h, x_m, x_p):
    """Eq. 3 voi lambda=0.5. sample_weight cua dataset DA BO (plan §4.1) -> lambda la
    trong so DUY NHAT. Batch giu cau truc bo ba 1:1:1 dung gia dinh cua paper."""
    det.train(); losses = []
    n = len(x_h)
    for i in range(0, n, DET_TRIPLETS):
        h, m, p = x_h[i:i + DET_TRIPLETS], x_m[i:i + DET_TRIPLETS], x_p[i:i + DET_TRIPLETS]
        texts = list(h) + list(m) + list(p)
        labels = torch.tensor([0] * len(h) + [1] * len(m) + [1] * len(p), device=DET_DEV)
        w = torch.tensor([1.0] * len(h) + [RADAR_LAMBDA] * (len(m) + len(p)),
                         device=DET_DEV, dtype=torch.float)
        enc = det_tok(texts, truncation=True, max_length=MAX_LEN, padding=True,
                      pad_to_multiple_of=8, return_tensors="pt").to(DET_DEV)
        with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            lg = det(**enc).logits
            loss = (F.cross_entropy(lg.float(), labels, reduction="none") * w).mean()
        det_opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(det_opt)
        torch.nn.utils.clip_grad_norm_(det.parameters(), 1.0)
        scaler.step(det_opt); scaler.update(); det_sched.step()
        losses.append(float(loss))
    return float(np.mean(losses)) if losses else float("nan")

def val_auroc():
    s = detector_scores(val_df["text"].tolist())
    return float(roc_auc_score(val_df["label"].values, s))

# ---------------------------- RESUME ----------------------------------------
# ppo.model thay vi policy: accelerate co the tra ve object khac sau prepare().
state = {"outer": 0, "best_auroc": -1.0, "n_bad_evals": 0, "monitor": []}
if (CKPT_DIR / "state.json").exists():
    state = json.loads((CKPT_DIR / "state.json").read_text())
    det.load_state_dict(torch.load(CKPT_DIR / "det.pt", map_location=DET_DEV))
    det_opt.load_state_dict(torch.load(CKPT_DIR / "det_opt.pt", map_location=DET_DEV))
    det_sched.load_state_dict(torch.load(CKPT_DIR / "det_sched.pt"))
    ppo.model.load_state_dict(torch.load(CKPT_DIR / "policy.pt", map_location=PARA_DEV))
    if (CKPT_DIR / "ppo_opt.pt").exists():
        ppo_opt.load_state_dict(torch.load(CKPT_DIR / "ppo_opt.pt", map_location=PARA_DEV))
    if (CKPT_DIR / "kl_ctl.json").exists():
        try:
            ppo.kl_ctl.value = json.loads((CKPT_DIR / "kl_ctl.json").read_text())["value"]
        except Exception:
            pass
    print("RESUME tu outer_step=%d (best_auroc=%.4f)" % (state["outer"], state["best_auroc"]), flush=True)

def save_resume():
    torch.save(det.state_dict(),        CKPT_DIR / "det.pt")
    torch.save(det_opt.state_dict(),    CKPT_DIR / "det_opt.pt")
    torch.save(det_sched.state_dict(),  CKPT_DIR / "det_sched.pt")
    torch.save(ppo.model.state_dict(),  CKPT_DIR / "policy.pt")
    torch.save(ppo_opt.state_dict(),    CKPT_DIR / "ppo_opt.pt")   # moment cua Adam -- mat la PPO giat lai
    try:
        (CKPT_DIR / "kl_ctl.json").write_text(json.dumps({"value": float(ppo.kl_ctl.value)}))
    except Exception:
        pass
    (CKPT_DIR / "state.json").write_text(json.dumps(state, indent=2, default=float))

# ---------------------------- GAME LOOP -------------------------------------
buf_q, buf_r, buf_s = [], [], []
rng = np.random.RandomState(SEED + state["outer"])
t_start = time.time()
print("\n" + "=" * 78)
print("BAT DAU GAME LOOP -- outer %d -> %d" % (state["outer"], MAX_OUTER_STEPS))
print("LUU Y: vai chuc step dau reward gan nhu HANG SO (D_phi con ngau nhien) -- DUNG, khong phai bug.")
print("=" * 78, flush=True)

stop = False
for outer in range(state["outer"], MAX_OUTER_STEPS):
    idx = rng.choice(len(pairs), size=min(DOCS_PER_STEP, len(pairs)), replace=False)
    batch = pairs.iloc[idx]
    x_h = batch["human_text"].tolist(); x_m = batch["ai_text"].tolist()

    # --- Step 2: paraphrase + reward ---
    x_p, qs, rs, owner = paraphrase_batch(x_m)
    p_ai = detector_scores(x_p)
    r_doc = 1.0 - p_ai                                   # Eq. 1: R = D_phi(x_p) = P(human)

    len_ratio = np.mean([len(b.split()) / max(1, len(a.split())) for a, b in zip(x_m, x_p)])
    buf_q += qs; buf_r += rs
    buf_s += [float(r_doc[o]) for o in owner]            # broadcast reward document -> cau

    # --- Step 2b: PPO khi du dung PPO_BATCH mau ---
    kl = float("nan"); n_ppo = 0; r_std = float("nan")
    while len(buf_q) >= PPO_BATCH:
        q, r, s = buf_q[:PPO_BATCH], buf_r[:PPO_BATCH], buf_s[:PPO_BATCH]
        buf_q, buf_r, buf_s = buf_q[PPO_BATCH:], buf_r[PPO_BATCH:], buf_s[PPO_BATCH:]
        R = torch.tensor(s, dtype=torch.float)
        r_std = float(R.std())
        if r_std < REWARD_STD_EPS:
            # A = (R-mean)/std khong xac dinh khi std~0 -> bo qua update nay.
            # KHONG phai doi method: chinh paper cung khong dinh nghia advantage o day.
            continue
        A = (R - R.mean()) / (R.std() + 1e-8)            # Algorithm 1 dong 14
        st = ppo.step(q, r, [a.to(PARA_DEV) for a in A])
        kl = float(st.get("objective/kl", float("nan"))); n_ppo += 1

    # --- Step 3: cap nhat detector (Eq. 3) ---
    det_loss = detector_update(x_h, x_m, x_p)

    row = {"outer": outer, "reward_mean": float(r_doc.mean()), "reward_std": r_std,
           "kl": kl, "n_ppo": n_ppo, "det_loss": det_loss, "len_ratio": len_ratio,
           "buf": len(buf_q), "elapsed_min": (time.time() - t_start) / 60}

    # --- Step 4: validation + chon phi_best / sigma_best ---
    if outer % EVAL_EVERY == 0 or outer == MAX_OUTER_STEPS - 1:
        a = val_auroc(); row["val_auroc"] = a
        if a > state["best_auroc"]:
            state["best_auroc"] = a; state["n_bad_evals"] = 0
            if not SMOKE:
                det.save_pretrained(OUT_DIR / "model"); det_tok.save_pretrained(OUT_DIR / "model")
                # sigma_best = paraphraser GHEP CAP voi phi_best (Algorithm 1 dong 28)
                ppo.model.save_pretrained(OUT_DIR / "paraphraser")
                para_tok.save_pretrained(OUT_DIR / "paraphraser")
            print("   -> phi_best MOI (AUROC val = %.4f), da luu ca sigma_best" % a, flush=True)
        else:
            state["n_bad_evals"] += 1
            if state["n_bad_evals"] >= EARLY_STOP_EVALS:
                print("   -> EARLY STOP: %d lan eval khong cai thien" % state["n_bad_evals"], flush=True)
                stop = True

    state["monitor"].append(row)
    print("[%4d] R %.3f (std %.3f) | kl %7.3f | ppo %d | det_loss %.4f | len %.3f | val %s | %.1f min"
          % (outer, row["reward_mean"], row["reward_std"], row["kl"], n_ppo, row["det_loss"],
             len_ratio, ("%.4f" % row["val_auroc"]) if "val_auroc" in row else "  -   ",
             row["elapsed_min"]), flush=True)

    state["outer"] = outer + 1
    if not SMOKE and (outer % SAVE_EVERY == 0 or stop):
        save_resume(); pd.DataFrame(state["monitor"]).to_csv(OUT_DIR / "monitor.csv", index=False)
    if stop:
        break

pd.DataFrame(state["monitor"]).to_csv(OUT_DIR / "monitor.csv", index=False)
if SMOKE:
    print("\nSMOKE xong. Kiem 4 gate o markdown cell roi chay FULL.")
    raise SystemExit(0)

# ---------------------------- EVAL CUOI -------------------------------------
# Load lai phi_best roi do bang DUNG block metric cua supervised.
print("\n" + "=" * 78); print("EVAL phi_best"); print("=" * 78, flush=True)
det = AutoModelForSequenceClassification.from_pretrained(OUT_DIR / "model").to(DET_DEV).eval()

for d in [val_df, ood_df, fp_df]:
    if d is not None:
        d["score"] = detector_scores(d["text"].tolist())

def tpr_at_fpr(y, s, fpr):
    y, s = np.asarray(y), np.asarray(s); neg = s[y == 0]
    if len(neg) == 0 or (y == 1).sum() == 0:
        return np.nan, np.nan
    thr = np.quantile(neg, 1 - fpr, method="higher")
    return float((s[y == 1] >= thr).mean()), float(thr)

def auroc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else np.nan

_, TAU = tpr_at_fpr(val_df["label"], val_df["score"], TARGET_FPR)
print("tau (FPR=1%% tren val) = %.4f" % TAU, flush=True)

def report_split(name, d):
    if d is None:
        return None
    y, s = d["label"].values, d["score"].values
    row = {"split": name, "n": len(d), "n_ai": int((y == 1).sum()), "auroc": auroc(y, s),
           "auprc": float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else np.nan}
    for f in [0.001, 0.01, 0.05]:
        row["tpr@fpr%s" % f] = tpr_at_fpr(y, s, f)[0]
    row["tpr@tau"] = float((s[y == 1] >= TAU).mean()) if (y == 1).any() else np.nan
    row["fpr@tau"] = float((s[y == 0] >= TAU).mean()) if (y == 0).any() else np.nan
    return row

rows = [r for r in [report_split("val", val_df), report_split("test_ood_model", ood_df),
                    report_split("test_fp", fp_df)] if r]

# ── ĐÁNH GIÁ ĐỐI-SHORTCUT trên TẬP MATCHED (make_matched_eval.py) ─────
# Đây là con số trả lời phản biện "0.99 của anh chỉ là đếm chữ với độ dài từ".
#
# Tập matched là tập con mà MỌI mô hình chỉ-đặc-trưng-bề-mặt đều về mức ngẫu nhiên —
# đã kiểm bằng hai mô hình để không lập luận vòng tròn (fit trên train, và fit LẠI
# out-of-fold trên chính tập con):
#     val            876 -> 296 dòng    surface 0.9049 -> 0.5027 / 0.5475
#     test_ood_model 3.028 -> 1.206     surface 0.8715 -> 0.5446 / 0.4663
# Nên nếu detector vẫn giữ AUROC cao ở đây thì phần đó KHÔNG đến từ đặc trưng bề mặt.
#
# Sàn để so: GBM 22 đặc trưng bề mặt đạt AUROC 0.8715 trên test_ood_model đầy đủ
# nhưng 0.5446 trên tập matched (surface_baseline.py). Khoảng cách giữa detector và
# sàn đó là phần detector thực sự đóng góp.
#
# Chấm lại trực tiếp trên text của tập matched, KHÔNG join theo chỉ số dòng: tập
# matched chỉ giữ ~1/3 dòng và thứ tự khác, join theo vị trí là sai âm thầm.
# Tra globals() thay vì gọi thẳng tên: nhánh supervised có predict_proba() và
# SMOKE_TEST, nhánh RADAR dùng detector_scores() và SMOKE. Gọi thẳng thì block này
# chạy được ở nhánh này và NameError ở nhánh kia — mà lỗi đó chỉ nổ ở CUỐI buổi
# train, sau khi đã tốn hết GPU-hour.
_score_fn = globals().get("predict_proba") or globals().get("detector_scores")
_smoke = bool(globals().get("SMOKE_TEST") or globals().get("SMOKE") or False)
assert _score_fn is not None, "không tìm thấy hàm chấm điểm (predict_proba/detector_scores)"

for _sp in ("val", "test_ood_model"):
    _mp = DATA_DIR / f"matched_eval_{_sp}_{DATASET_TAG}.parquet"
    if not _mp.exists():
        print(f"  (thiếu {_mp.name} -> bỏ qua đánh giá matched cho {_sp})", flush=True)
        continue
    _m = pd.read_parquet(_mp)
    _m["score"] = _score_fn(_m["text"].tolist())
    _row = report_split(f"{_sp}_matched", _m)
    if _row:
        rows.append(_row)
        print(f"  matched {_sp}: n={len(_m)} AUROC={_row['auroc']:.4f}", flush=True)
    if not _smoke:
        _keep = [c for c in ("sample_id", "paper_id", "label", "text_role",
                             "surface_propensity", "score") if c in _m.columns]
        _m[_keep].to_parquet(OUT_DIR / f"pred_matched_{_sp}.parquet", index=False)
metrics_df = pd.DataFrame(rows).set_index("split")
pd.set_option("display.width", 200, "display.max_columns", 20)
print(metrics_df.round(4).to_string(), flush=True)

metrics_out = {"run": RUN_NAME, "model": DETECTOR, "method": "radar", "eval_mode": "inductive",
               "paraphraser": PARAPHRASER, "para_unit": PARA_UNIT,
               "tau": TAU, "target_fpr": TARGET_FPR, "seed": SEED, "n_seeds": 1,
               "sample_weight_used": False, "world_size": 1,
               "hparams": {"radar_lambda": RADAR_LAMBDA, "gamma_entropy": GAMMA_ENT,
                           "ppo_batch_sentences": PPO_BATCH, "ppo_epochs": PPO_EPOCHS,
                           "docs_per_step": DOCS_PER_STEP, "top_k": TOP_K, "top_p": TOP_P,
                           "init_kl_coef": INIT_KL_COEF, "kl_target": KL_TARGET,
                           "outer_steps_run": state["outer"], "best_val_auroc": state["best_auroc"]},
               "metrics": metrics_df.reset_index().to_dict("records")}
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2, default=float))
for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    if d is not None:
        d[["paper_id", "label", "ai_degree", "generator_family", "score"]].to_parquet(
            OUT_DIR / ("pred_%s.parquet" % nm), index=False)
print("da luu artifacts ->", OUT_DIR, flush=True)
print("!! GIU LAI %s/paraphraser -- do la 'seen paraphraser', can cho attack_eval_plan." % OUT_DIR,
      flush=True)

## 2. SMOKE — 3 outer step

Đây **chỉ kiểm đường ống**, không kiểm chất lượng học. 4 gate:

| Gate | Kỳ vọng |
|---|---|
| Chạy hết 3 step, không OOM | ✅ |
| `len` (tỉ lệ độ dài `x_p`/`x_m`) | **∈ [0.85, 1.15]** — gate chống shortcut độ dài |
| `det_loss` | hữu hạn, không NaN |
| `kl` | hữu hạn hoặc `nan` (nan = chưa step PPO lần nào vì `std(reward)` còn ~0 → **bình thường ở step đầu**) |

> `R std ≈ 0` ở SMOKE là **đúng** — `Dφ` mới chạy 3 step. Gate `std(reward) > 0` chỉ đánh giá được
> ở đầu FULL run, sau khi `AUROC(val)` bắt đầu tăng.

In [ ]:
!SMOKE=1 python train_radar.py

## 3. FULL RUN

**1 seed** (ghi rõ khi báo cáo). Không fix cứng số epoch — `MAX_OUTER_STEPS=3000` chỉ là trần,
**early-stop theo AUROC(val)** mới quyết định điểm dừng (§4.1 Implementation Details: *"train the models until the validation
loss converges"*).

> **Hết giờ Kaggle thì cứ để nó dừng.** Cell này tự resume: `ckpt_resume/` lưu
> `{Dφ, Gσ, optimizer, scheduler, outer_step, φ_best}` mỗi 40 outer step.
> Session sau **Save Version → Run All lại chính cell này** là chạy tiếp từ chỗ dừng.

**Theo dõi trong lúc chạy** (thứ tự xuất hiện):
1. `det_loss` giảm và `val` tăng — `Dφ` đang học (không phụ thuộc reward).
2. Sau đó `R std` **rời khỏi 0** → reward bắt đầu có tín hiệu → `ppo` > 0 → PPO thực sự chạy.
3. `kl` bounded (adaptive KL kéo về `target=6.0`), `len` giữ trong [0.85, 1.15].

Nếu `R std` **không bao giờ** rời 0 dù `val` đã cao: paraphrase quá gần bản gốc → quay lại
`preflight_paraphraser.ipynb`, đổi `PARA_UNIT` hoặc checkpoint.

In [ ]:
!python train_radar.py

In [ ]:
# ============ Bao cao + figure ============
import json, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

OUT = Path("artifacts_radar_xlm_seed42")
m = json.loads((OUT / "metrics.json").read_text())
print("=== RADAR (1 seed) | paraphraser: %s | don vi: %s ==="
      % (m["paraphraser"], m["para_unit"]))
print("outer step da chay: %d | best AUROC(val): %.4f"
      % (m["hparams"]["outer_steps_run"], m["hparams"]["best_val_auroc"]))
print(pd.DataFrame(m["metrics"]).set_index("split").round(4).to_string())

mon = pd.read_csv(OUT / "monitor.csv")
fig, ax = plt.subplots(2, 2, figsize=(13, 7))
ax[0, 0].plot(mon["outer"], mon["reward_mean"]); ax[0, 0].set_title("reward mean = P(human | x_p)")
ax[0, 1].plot(mon["outer"], mon["reward_std"]); ax[0, 1].axhline(1e-3, ls="--", c="r")
ax[0, 1].set_title("reward std (phai roi khoi 0 thi PPO moi hoc)")
v = mon.dropna(subset=["val_auroc"]) if "val_auroc" in mon else mon.iloc[0:0]
if len(v):
    ax[1, 0].plot(v["outer"], v["val_auroc"], marker="o")
ax[1, 0].set_title("AUROC(val) cua D_phi")
ax[1, 1].plot(mon["outer"], mon["len_ratio"])
ax[1, 1].axhline(0.85, ls="--", c="r"); ax[1, 1].axhline(1.15, ls="--", c="r")
ax[1, 1].set_title("len(x_p)/len(x_m) -- gate chong shortcut do dai")
for a in ax.ravel():
    a.set_xlabel("outer step")
plt.tight_layout(); plt.savefig("radar_monitor.png", dpi=140); plt.show()

print("\n!! sigma_best ('seen paraphraser') nam o: %s" % (OUT / "paraphraser"))
print("   GIU LAI -- attack_eval_plan.md §7 can no de tai lap Table 2 cua paper.")